In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

print("Training data shape:", train.shape)
print("Test data shape:", test.shape)

In [ ]:
train.head()
train.shape
train.columns
train.info()
train.isnull().sum()

Percent the missing data compare to total data

In [ ]:
train.isnull().mean() * 100

the scatter of the data

In [ ]:
train.describe()

In [ ]:
train["Age"].min(), train["Age"].max()

In [ ]:
train["Age"].hist()

In [ ]:
train["Age"].skew()

In [ ]:
train["Fare"].hist()

In [ ]:
train["Survived"].value_counts(normalize=True)

relationship between feature and target

In [ ]:
train.groupby("Sex")["Survived"].mean()
train.groupby("Pclass")["Survived"].mean()
train.groupby("Survived")["Age"].mean()

In [ ]:
train["AgeGroup"] = pd.cut(
    train["Age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=["Child", "Teen", "Young Adult", "Adult", "Senior"]
)

train.groupby("AgeGroup", observed=True)["Survived"].mean()

In [ ]:
train.groupby("SibSp")["Survived"].mean()

In [ ]:

train.groupby("Parch")["Survived"].mean()

In [ ]:
train["Parch"].value_counts().sort_index()

In [ ]:
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
train.groupby("FamilySize")["Survived"].mean()

In [ ]:
train["IsAlone"] = (train["FamilySize"] == 1).astype(int)

In [ ]:
train.groupby("IsAlone")["Survived"].mean()

In [ ]:
train.groupby("Embarked")["Survived"].mean()

In [ ]:
train.groupby("Survived")["Fare"].mean()

Establishing of the first model

In [ ]:
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
    "FamilySize",
    "IsAlone"
]

In [ ]:
X = train[features]
y = train["Survived"]

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
numeric_features = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "FamilySize",
    "IsAlone"
]

categorical_features = [
    "Sex",
    "Embarked"
]

In [ ]:
from sklearn.impute import SimpleImputer
numeric_imputer = SimpleImputer(strategy="median")

In [ ]:
numeric_imputer.fit(X[numeric_features])

In [ ]:
x_numeric_imputed = numeric_imputer.transform

In [ ]:
X_numeric_imputed = numeric_imputer.transform(X[numeric_features])

In [ ]:
np.isnan(X_numeric_imputed).sum()

In [ ]:
from sklearn.preprocessing import OneHotEncoder
categorical_imputer = SimpleImputer(strategy="most_frequent")

In [ ]:
categorical_imputer.fit(X[categorical_features])

X_categorical_imputed = categorical_imputer.transform(
    X[categorical_features]
)

In [ ]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

In [ ]:
encoder.fit(X_categorical_imputed)

In [ ]:
X_categorical_encoded = encoder.transform(
    X_categorical_imputed
)

In [ ]:
encoder.get_feature_names_out(categorical_features)

In [ ]:
X_processed = np.concatenate(
    [X_numeric_imputed, X_categorical_encoded],
    axis=1
)
X_processed.shape
X_processed[:3]


In [ ]:
X = train[features]
y = train["Survived"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print(X_train.shape)
print(X_valid.shape)
print(y_train.shape)
print(y_valid.shape)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

In [ ]:
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)
from sklearn.linear_model import LogisticRegression

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)
model.fit(X_train, y_train)

In [ ]:
valid_predictions = model.predict(X_valid)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_valid, valid_predictions)

print("Validation accuracy:", accuracy)

In [ ]:
train["Title"] = train["Name"].str.extract(r",\s*([^.]*)\.")

In [ ]:
train["Title"].value_counts()

In [ ]:
train["Title"] = train["Title"].replace({
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"
})
common_titles = ["Mr", "Miss", "Mrs", "Master"]

train["Title"] = train["Title"].apply(
    lambda x: x if x in common_titles else "Rare"
)
train["Title"].value_counts()

In [ ]:
categorical_features = [
    "Sex",
    "Embarked",
    "Title"
]
model.fit(X_train, y_train)

In [ ]:
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
    "FamilySize",
    "IsAlone",
    "Title"
]

X = train[features]
y = train["Survived"]
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
train["Title"].value_counts()

In [ ]:
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
    "FamilySize",
    "IsAlone",
    "Title"
]

X = train[features]
y = train["Survived"]

In [ ]:
categorical_features = [
    "Sex",
    "Embarked",
    "Title"
]
numeric_features = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "FamilySize",
    "IsAlone"
]
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
random_state=42

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

model.fit(X_train, y_train)

valid_predictions = model.predict(X_valid)

accuracy = accuracy_score(y_valid, valid_predictions)

print("Validation accuracy:", accuracy)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            random_state=42
        ))
    ]
)

In [ ]:
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_valid)
rf_accuracy = accuracy_score(y_valid, rf_predictions)

print("Random Forest accuracy:", rf_accuracy)

In [ ]:
print("Random Forest accuracy:", rf_accuracy)

In [ ]:
RandomForestClassifier(
    n_estimators=200,
    random_state=42
)
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            max_depth=5,
            min_samples_split=5,
            min_samples_leaf=2,
            random_state=42
        ))
    ]
)

In [ ]:
rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_valid)

rf_accuracy = accuracy_score(y_valid, rf_predictions)

print("Tuned Random Forest accuracy:", rf_accuracy)

In [ ]:
from xgboost import XGBClassifier
xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            eval_metric="logloss"
        ))
    ]
)
xgb_model.fit(X_train, y_train)
xgb_predictions = xgb_model.predict(X_valid)
xgb_accuracy = accuracy_score(y_valid, xgb_predictions)

print("XGBoost accuracy:", xgb_accuracy)

In [ ]:
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1

test["IsAlone"] = (test["FamilySize"] == 1).astype(int)

In [ ]:
test["Title"] = test["Name"].str.extract(r",\s*([^.]*)\.")

In [ ]:
test["Title"] = test["Title"].replace({
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"
})

common_titles = ["Mr", "Miss", "Mrs", "Master"]

test["Title"] = test["Title"].apply(
    lambda x: x if x in common_titles else "Rare"
)

In [ ]:
X = train[features]
y = train["Survived"]

model.fit(X, y)

In [ ]:
X_test = test[features]
test_predictions = model.predict(X_test)

In [ ]:
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
    "FamilySize",
    "IsAlone",
    "Title"
]

In [ ]:
categorical_features = [
    "Sex",
    "Embarked",
    "Title"
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

In [ ]:
X = train[features]
y = train["Survived"]

model.fit(X, y)

In [ ]:
X_test = test[features]

test_predictions = model.predict(X_test)
len(test_predictions)

In [ ]:
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": test_predictions
})

In [ ]:
submission.head()

In [ ]:
submission.shape

In [ ]:
submission.to_csv("submission.csv", index=False)

In [ ]:
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
    "FamilySize",
    "IsAlone",
    "Title"
]

X = train[features]
y = train["Survived"]

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print(scores)
print("Mean:", scores.mean())
print("Std:", scores.std())